# Cross-Dataset Generalisation — Colab Runner

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save  
**Estimated time:** DistilBERT × 3 datasets × 1 seed ≈ 2–2.5 hours on T4  

Run cells top-to-bottom. Each cell prints its result before moving on.

In [5]:
!pip install "accelerate>=0.33.0" --force-reinstall -q
!pip install "transformers>=4.41.0" --force-reinstall -q
print("Done.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.5/671.5 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 0. Clone repo and set working directory

In [1]:
import os

# ── FILL THIS IN ──────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/adnankhalil22/fake-news-generalization.git"
# ──────────────────────────────────────────────────────────────────────────────

REPO_DIR    = "/content/fake-news-generalization"
# The git repo is rooted at the user home dir; project files live in this subpath.
PROJECT_DIR = os.path.join(REPO_DIR, "OneDrive", "Desktop", "final research")

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes")
    !git -C {REPO_DIR} pull

%cd {PROJECT_DIR}
print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))

Repo already cloned — pulling latest changes
Already up to date.
/content/fake-news-generalization/OneDrive/Desktop/final research
Working directory: /content/fake-news-generalization/OneDrive/Desktop/final research
Contents: ['notebooks', 'README.md', 'DATASETS.md', 'src', 'requirements.txt', '.gitignore', 'RESULTS.md', 'results']


## 1. Install dependencies

In [2]:
# Install pinned packages from requirements.txt
!pip install -r requirements.txt -q

# Colab's pre-compiled packages (sklearn, scipy) need NumPy 2.x.
# requirements.txt pins numpy==1.26.4 for local reproducibility, but that
# downgrades Colab's NumPy and causes binary incompatibility errors.
# Upgrade it back so Colab's C extensions can load correctly.
!pip install "numpy>=2.0.0" -q

# spaCy model (only needed for masking experiments, NOT for DistilBERT training)
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
    capture_output=True, text=True
)
if r.returncode == 0:
    print("spaCy model: installed.")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "spacy", "--upgrade", "-q"])
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

print("\nInstallation complete.")
print("=" * 55)
print("ACTION REQUIRED: Runtime → Restart runtime")
print("Then run from Cell 2 onwards (skip this cell).")
print("=" * 55)

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'

Installation complete.
ACTION REQUIRED: Runtime → Restart runtime
Then run from Cell 2 onwards (skip this cell).


## 2. Verify GPU and dataset access

In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU — DistilBERT training will be very slow.")
    print("Go to Runtime > Change runtime type > T4 GPU")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

GPU available: True
GPU: Tesla T4


In [3]:
from src.data import get_dataset, DATASETS

print("Dataset access check:")
all_ok = True
for ds in DATASETS:
    try:
        df = get_dataset(ds, 'train')
        print(f"  OK  {ds}: n={len(df)}  fake={df.label.mean():.1%}")
    except Exception as e:
        print(f"  ERR {ds}: {e}")
        all_ok = False

if all_ok:
    print("\nAll datasets loaded. Ready to train.")
else:
    print("\nFix the errors above before continuing.")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/__init__.py:16: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.sparse import issparse
2026-06-01 07:29:55  src.data  INFO  Loading LIAR split=train
INFO:src.data:Loading LIAR split=train


Dataset access check:


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
2026-06-01 07:29:57  src.data  INFO  LIAR: HuggingFace unavailable — downloading TSV files from UCSB
INFO:src.data:LIAR: HuggingFace unavailable — downloading TSV files from UCSB
2026-06-01 07:29:57  src.data  INFO  LIAR: downloading zip from https://www.cs.ucsb.edu/~william/data/liar_dataset.zip
INFO:src.data:LIAR: downloading zip from https://www.cs.ucsb.edu/~william/data/liar_dataset.zip
2026-06-01 07:29:58  src.data  INFO  LIAR: extracted 'train.tsv' → 'data/raw/liar/train

  OK  liar: n=10269  fake=43.8%


Repo card metadata block was not found. Setting CardData to empty.


...

Generating test split:   0%|          | 0/8117 [00:00<?, ? examples/s]

2026-06-01 07:30:05  src.data  INFO  welfake/hf train: n=24353  real=13195  fake=11158
INFO:src.data:welfake/hf train: n=24353  real=13195  fake=11158
2026-06-01 07:30:05  src.data  INFO  Subsampled welfake/train to 20000 examples (cap=20000)
INFO:src.data:Subsampled welfake/train to 20000 examples (cap=20000)
2026-06-01 07:30:05  src.data  INFO  COVID: cached files not found — downloading from GitHub
INFO:src.data:COVID: cached files not found — downloading from GitHub
2026-06-01 07:30:05  src.data  INFO  COVID: downloading 'Constraint_Train.csv' from https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/Constraint_Train.csv
INFO:src.data:COVID: downloading 'Constraint_Train.csv' from https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/Constraint_Train.csv


  OK  welfake: n=20000  fake=45.8%


2026-06-01 07:30:06  src.data  INFO  COVID: saved to 'data/raw/covid/Constraint_Train.csv'
INFO:src.data:COVID: saved to 'data/raw/covid/Constraint_Train.csv'
2026-06-01 07:30:06  src.data  INFO  COVID: downloading 'Constraint_Val.csv' from https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/Constraint_Val.csv
INFO:src.data:COVID: downloading 'Constraint_Val.csv' from https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/Constraint_Val.csv
2026-06-01 07:30:06  src.data  INFO  COVID: saved to 'data/raw/covid/Constraint_Val.csv'
INFO:src.data:COVID: saved to 'data/raw/covid/Constraint_Val.csv'
2026-06-01 07:30:06  src.data  INFO  COVID: downloading 'english_test_with_labels.csv' from https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/english_test_with_labels.csv
INFO:src.data:COVID: downloading 'english_test_with_labels.csv' from https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/english_test_with_labels.csv
2026-06-

  OK  covid: n=6420  fake=47.7%

All datasets loaded. Ready to train.


## 3. Train DistilBERT on each dataset

One seed (42) per dataset. Each dataset takes ~40 min on T4.  
**Do not close this tab** — Colab disconnects after ~90 min of inactivity.  
If it disconnects, the Trainer saves checkpoints each epoch; you can resume.

In [4]:
from src.train import train_distilbert

SEED = 42

for ds in DATASETS:
    print(f"\n{'='*55}")
    print(f"Training DistilBERT on {ds.upper()}  (seed={SEED})")
    print(f"{'='*55}")
    metrics = train_distilbert(ds, seed=SEED)
    print(f"  val macro-F1 : {metrics.get('eval_macro_f1', 'n/a')}")
    print(f"  val accuracy : {metrics.get('eval_accuracy', 'n/a')}")

print("\nAll three DistilBERT models trained.")

2026-06-01 07:30:11  src.train  INFO  [distilbert] training on liar  seed=42
INFO:src.train:[distilbert] training on liar  seed=42



Training DistilBERT on LIAR  (seed=42)


RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
cannot import name 'LocalEntryNotFoundError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)

## 4. Build the 3x3 evaluation matrix

In [ ]:
from src.evaluate import build_matrix

distilbert_matrix = build_matrix('distilbert', seeds=[SEED])
print("\nMatrix saved to results/matrices/distilbert_f1_matrix.csv")

In [ ]:
from IPython.display import Image, display
display(Image('results/figures/distilbert_f1_heatmap.png'))

## 5. Save results to Google Drive

Run this so you don't lose results if Colab resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dest = '/content/drive/MyDrive/fake-news-results'
os.makedirs(dest, exist_ok=True)
shutil.copytree('results', os.path.join(dest, 'results'), dirs_exist_ok=True)
print(f"Results saved to Google Drive: {dest}")

## 6. Print final matrix (paste this output back to Claude)

Run this last cell and copy the full output.

In [ ]:
import pandas as pd

print("\n=== DistilBERT Macro-F1 Matrix (rows=train, cols=test) ===")
df = pd.read_csv('results/matrices/distilbert_f1_matrix.csv', index_col=0)
print(df.to_string())

print("\n=== LogReg Macro-F1 Matrix (for comparison) ===")
df2 = pd.read_csv('results/matrices/logreg_f1_matrix.csv', index_col=0)
print(df2.to_string())

print("\n--- Copy everything above this line and paste to Claude ---")